# YatraLok — AI/ML Risk Prediction Model Training

This notebook trains baseline machine learning models to predict tourist risk levels (LOW, MEDIUM, HIGH) based on movement analysis and location context.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 2: Load Dataset

In [2]:
data_path = 'data/dataset.csv' if os.path.exists('data/dataset.csv') else 'dataset.csv'
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print("\nTarget Class Distribution:")
print(df['risk_level'].value_counts())

Dataset Shape: (27000, 19)

Target Class Distribution:
LOW       22404
MEDIUM     4232
HIGH        364
Name: risk_level, dtype: int64


## Step 3: Feature Selection & Partitioning

In [3]:
FEATURE_COLS = [
    'distance_from_danger_zone',
    'distance_from_restricted_zone',
    'speed',
    'time_of_day',
    'duration_in_remote_area',
    'movement_frequency',
    'route_deviation',
    'stationary_duration',
    'zone_entry_history',
    'previous_incident_context'
]

train_df = df[df['dataset_split'] == 'train']
val_df = df[df['dataset_split'] == 'validation']
test_df = df[df['dataset_split'] == 'test']

X_train, y_train = train_df[FEATURE_COLS], train_df['risk_level']
X_val, y_val = val_df[FEATURE_COLS], val_df['risk_level']
X_test, y_test = test_df[FEATURE_COLS], test_df['risk_level']

print(f"Features selected: {len(FEATURE_COLS)}")
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")

Features selected: 10
Training samples   : 18900
Validation samples : 5400
Test samples       : 2700


## Step 4: Feature Scaling

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features scaled with StandardScaler (fitted on train set only).")

Features scaled with StandardScaler (fitted on train set only).


## Step 5: Model 1 — Logistic Regression Baseline

In [5]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_val_scaled)
print(f"Logistic Regression -> Validation Accuracy: {accuracy_score(y_val, lr_preds)*100:.2f}%, F1-Score: {f1_score(y_val, lr_preds, average='weighted'):.4f}")

Logistic Regression -> Validation Accuracy: 77.24%, F1-Score: 0.8020


## Step 6: Model 2 — Decision Tree Classifier Baseline

In [6]:
dt_model = DecisionTreeClassifier(max_depth=8, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

dt_preds = dt_model.predict(X_val)
print(f"Decision Tree       -> Validation Accuracy: {accuracy_score(y_val, dt_preds)*100:.2f}%, F1-Score: {f1_score(y_val, dt_preds, average='weighted'):.4f}")

Decision Tree       -> Validation Accuracy: 95.09%, F1-Score: 0.9530


## Step 7: Model 3 — Random Forest Classifier

In [7]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_val)
print(f"Random Forest       -> Validation Accuracy: {accuracy_score(y_val, rf_preds)*100:.2f}%, F1-Score: {f1_score(y_val, rf_preds, average='weighted'):.4f}")

Random Forest       -> Validation Accuracy: 96.67%, F1-Score: 0.9662


## Step 8: Final Model Evaluation on Test Set

In [8]:
best_model = rf_model
test_preds = best_model.predict(X_test)

print("=== FINAL TEST EVALUATION RESULTS ===")
print(f"Accuracy  : {accuracy_score(y_test, test_preds) * 100:.2f}%")
print(f"Precision : {precision_score(y_test, test_preds, average='weighted'):.4f}")
print(f"Recall    : {recall_score(y_test, test_preds, average='weighted'):.4f}")
print(f"F1-Score  : {f1_score(y_test, test_preds, average='weighted'):.4f}")
print("\nConfusion Matrix (Actual Rows: LOW, MEDIUM, HIGH | Predicted Columns):")
print(confusion_matrix(y_test, test_preds, labels=['LOW', 'MEDIUM', 'HIGH']))
print("\nClassification Report:")
print(classification_report(y_test, test_preds))

=== FINAL TEST EVALUATION RESULTS ===
Accuracy  : 97.41%
Precision : 0.9745
Recall    : 0.9741
F1-Score  : 0.9724

Confusion Matrix (Actual Rows: LOW, MEDIUM, HIGH | Predicted Columns):
[[2158   16    0]
 [  29  454    0]
 [   0   25   18]]

Classification Report:
              precision    recall  f1-score   support

        HIGH       1.00      0.42      0.59        43
         LOW       0.99      0.99      0.99      2174
      MEDIUM       0.92      0.94      0.93       483

    accuracy                           0.97      2700
   macro avg       0.97      0.78      0.84      2700
weighted avg       0.97      0.97      0.97      2700



## Step 9: Save Model Artifact

In [9]:
model_payload = {
    'model': best_model,
    'features': FEATURE_COLS,
    'classes': best_model.classes_.tolist()
}

joblib.dump(model_payload, 'model.pkl')
print("Model successfully saved to model.pkl")

Model successfully saved to model.pkl


## Step 10: Live Inference Test

In [10]:
sample_input = pd.DataFrame([{
    'distance_from_danger_zone': 45.0,
    'distance_from_restricted_zone': 120.0,
    'speed': 2.5,
    'time_of_day': 23,
    'duration_in_remote_area': 12.0,
    'movement_frequency': 0.8,
    'route_deviation': 250.0,
    'stationary_duration': 0.0,
    'zone_entry_history': 1.0,
    'previous_incident_context': 450.0
}])

pred_level = best_model.predict(sample_input)[0]
probs = best_model.predict_proba(sample_input)[0]
class_idx = list(best_model.classes_).index(pred_level)
risk_score = float(probs[class_idx])

print("Sample Prediction Output:")
import json
print(json.dumps({
    "risk_score": round(risk_score, 2),
    "risk_level": pred_level,
    "reason": "Risk indicators: tourist is within 45m of a danger zone; route deviation of 250m from planned corridor; sustained stay of 12 mins in remote area; late-night movement in isolated/sensitive proximity."
}, indent=2))

Sample Prediction Output:
{
  "risk_score": 0.85,
  "risk_level": "HIGH",
  "reason": "Risk indicators: tourist is within 45m of a danger zone; route deviation of 250m from planned corridor; sustained stay of 12 mins in remote area; late-night movement in isolated/sensitive proximity."
}
